In [0]:
%run ./00_config

In [0]:
limit = 100
offset = 0
all_countries = []

countries_configs = {
    "endpoint": "/v1/mds-references/countries",
    "root": "CountryResource",
    "list": "Countries",
    "item": "Country"
}

endpoint = countries_configs["endpoint"]
root_key = countries_configs["root"]
list_key = countries_configs["list"]
item_key = countries_configs["item"]

meta_block = {}

while True:
    url = f"{base_url}{endpoint}?limit={limit}&offset={offset}"
    print(f"sending request: {url}")
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.json()
        country_resource = data.get("CountryResource", {})
        if offset == 0:
            meta_block = country_resource.get("Meta", {})
            expected_total = meta_block.get("TotalCount", 0)
            print(f"TotalCount expected: {expected_total}")
        countries_dict = country_resource.get("Countries", {})
        current_page_items = countries_dict.get("Country", [])
        if not current_page_items:
            break
        if isinstance(current_page_items, dict):
            current_page_items = [current_page_items]
        all_countries.extend(current_page_items)
        print(f"succeed {offset} to {offset + len(current_page_items)}")
        offset += limit
        time.sleep(0.5)
    else:
        raise Exception(f"Error: offset = {offset}, Code: {response.status_code}")
inner_content = {item_key: all_countries}
final_structured_data = {
    root_key: {
        list_key: inner_content,
        "Meta": meta_block
    }
}
file_path = f"{volume_path}raw_countries_all.json"
with open(file_path, "w") as f:
    json.dump(final_structured_data, f)
print("Done")